In [41]:
import multiprocessing

from cs336_basics.pretokenization_example import find_chunk_boundaries
from cs336_basics.bpe import train_bpe
from tqdm import tqdm

special_tokens = ["<|endoftext|>"]
file_path = "../data/owt_valid.txt"
num_processes = multiprocessing.cpu_count()

In [42]:
from collections import defaultdict

from cs336_basics.pretokenize import pre_tokenize_parallel

print("Pretokening .......")
token_counter = pre_tokenize_parallel(file_path, special_tokens)

Pretokening .......


100%|██████████| 8/8 [00:22<00:00,  2.81s/it]


In [43]:
vocab_size = 300
merges = []
vocab = [bytes([i]) for i in range(256)]
vocab += [s.encode("utf-8") for s in special_tokens]
init_vocab_len = len(vocab)

token_pair_set = dict()
pair_counter = defaultdict(int)
for token, count in token_counter.items():
    s = set()
    for i in range(len(token) - 1):
        pair = token[i : i + 2]
        pair_counter[pair] += count
        s.add(pair)
    token_pair_set[token] = s

In [46]:
def train():
    print("Training BPE .......")
    for epoch in tqdm(range(init_vocab_len + 1, vocab_size)):
        max_pair = max(pair_counter, key=lambda p: (pair_counter[p], p))
        mp0, mp1 = max_pair
        merged_pair = mp0 + mp1
        merges.append(max_pair)

        for token, count in list(token_counter.items()):
            if max_pair not in token_pair_set[token]:
                continue   # no occurrence — token stays unchanged in dict
            modify = False
            new_token = []
            i = 0
            n = len(token)
            while i < n:
                if i + 1 < n and token[i] == mp0 and token[i + 1] == mp1:
                    modify = True
                    if new_token:                       # left-neighbor update
                        left = new_token[-1]
                        lp = (left, mp0)
                        c = pair_counter[lp] - count
                        if c:
                            pair_counter[lp] = c
                        else:
                            del pair_counter[lp]
                        pair_counter[(left, merged_pair)] += count
                    if i + 2 < n:                       # right-neighbor update
                        right = token[i + 2]
                        rp = (mp1, right)
                        c = pair_counter[rp] - count
                        if c:
                            pair_counter[rp] = c
                        else:
                            del pair_counter[rp]
                        pair_counter[(merged_pair, right)] += count
                    new_token.append(merged_pair)
                    i += 2
                else:
                    new_token.append(token[i])
                    i += 1

            # ── in-place update (no new_counter) ──────────────────────────
            if modify:
                new_key = tuple(new_token)
                del token_counter[token]
                del token_pair_set[token]
                token_counter[new_key] = token_counter.get(new_key, 0) + count
                token_pair_set[new_key] = {
                    (new_token[j], new_token[j + 1]) for j in range(len(new_token) - 1)
                }

        del pair_counter[max_pair]   # max_pair gone from all tokens; remove stale count
        vocab.append(merged_pair)

In [ ]:
%lprun -f train train()

Training BPE .......


 90%|█████████ | 38/42 [00:37<00:04,  1.00s/it]

In [ ]:
import heapq

class HeapDict:
    """
    Max-heap + dict hybrid.

    Keys  : any hashable (e.g. tuple[bytes, ...])
    Values: int

    Complexity:
      __setitem__ / update  — O(log n) amortized  (lazy deletion)
      __getitem__           — O(1)
      pop_max / peek_max    — O(log n) amortized
      __contains__          — O(1)
    """

    def __init__(self):
        self._map: dict = {}          # key -> current value
        self._heap: list = []         # (-value, key) min-heap acting as max-heap

    # ── dict-like interface ────────────────────────────────────────────────

    def __setitem__(self, key, value: int) -> None:
        """Insert or update. Old heap entry becomes stale and is skipped lazily."""
        self._map[key] = value
        heapq.heappush(self._heap, (-value, key))

    def __getitem__(self, key) -> int:
        return self._map[key]

    def __contains__(self, key) -> bool:
        return key in self._map

    def __delitem__(self, key) -> None:
        """O(1) logical delete; stale heap entry is purged on next pop/peek."""
        del self._map[key]

    def __len__(self) -> int:
        return len(self._map)

    def __iter__(self):
        return iter(self._map)

    def items(self):
        return self._map.items()

    # ── heap interface ─────────────────────────────────────────────────────

    def _purge_stale(self) -> None:
        """Drop heap entries whose value no longer matches the dict."""
        while self._heap:
            neg_val, key = self._heap[0]
            if key not in self._map or self._map[key] != -neg_val:
                heapq.heappop(self._heap)
            else:
                break

    def peek_max(self):
        """Return (key, value) for the largest value without removing."""
        self._purge_stale()
        if not self._heap:
            raise KeyError("peek on empty HeapDict")
        neg_val, key = self._heap[0]
        return key, -neg_val

    def pop_max(self):
        """Remove and return (key, value) for the largest value."""
        self._purge_stale()
        if not self._heap:
            raise KeyError("pop from empty HeapDict")
        neg_val, key = heapq.heappop(self._heap)
        del self._map[key]
        return key, -neg_val


# ── quick smoke test ───────────────────────────────────────────────────────
hd = HeapDict()
hd[(b'a', b'b')] = 5
hd[(b'c', b'd')] = 2
hd[(b'e', b'f')] = 8

hd[(b'a', b'b')] = 10         # update — re-heapifies lazily

print(hd.peek_max())           # ((b'a', b'b'), 10)
print(hd.pop_max())            # ((b'a', b'b'), 10)
print(hd.pop_max())            # ((b'e', b'f'), 8)

In [9]:
!pip install line_profiler

In [10]:
%load_ext line_profiler